In [3]:
import os
import time
import pandas as pd
import pubchempy as pcp
from tqdm import tqdm
from chembl_webresource_client.new_client import new_client

In [4]:
df = pd.read_csv("ctod_data.csv", low_memory=False)

# 2. Expand drug names from semi-colon list
df_expanded = df[['nct_id', 'drugs']].dropna()
df_expanded['drug_name'] = df_expanded['drugs'].astype(str).str.lower().str.split(';')
df_expanded = df_expanded.explode('drug_name')
df_expanded['drug_name'] = df_expanded['drug_name'].str.strip()
df_expanded = df_expanded[df_expanded['drug_name'].notna() & (df_expanded['drug_name'] != '')]

# 3. Filter to known drugs using ChEMBL
molecule = new_client.molecule
known_drugs = molecule.filter(max_phase__gte=1).only(['pref_name'])
known_drug_names = set(m['pref_name'].lower() for m in known_drugs if m['pref_name'])
df_expanded = df_expanded[df_expanded['drug_name'].isin(known_drug_names)]

# 4. Unique drug list
drug_names = df_expanded['drug_name'].dropna().unique()


In [5]:
# 5. Caching setup
CACHE_FILE = "partial_drug_info.csv"
if os.path.exists(CACHE_FILE):
    cached = pd.read_csv(CACHE_FILE)
    done = set(cached['drug_name'].dropna().str.lower())
    results = cached.to_dict('records')
    print(f"Loaded {len(done)} cached drugs")
else:
    results = []
    done = set()

mechanism = new_client.mechanism

In [6]:
# 6. Enrichment function
def get_drug_info(name):
    data = {'drug_name': name}

    # PubChem
    try:
        compound = pcp.get_compounds(name, 'name')[0]
        data['smiles'] = compound.canonical_smiles
        data['iupac'] = compound.iupac_name
        data['molecular_weight'] = compound.molecular_weight
        data['synonyms'] = "; ".join(compound.synonyms[:5]) if compound.synonyms else None
    except Exception as e:
        print(f"[PubChem fail] {name}: {e}")
        data['smiles'] = data['iupac'] = data['molecular_weight'] = data['synonyms'] = data['description'] = None

    # ChEMBL
    try:
        search = molecule.filter(pref_name__iexact=name)
        if not search:
            search = molecule.filter(molecule_synonyms__iexact=name)

        chembl_id = search[0]['molecule_chembl_id']
        mol = molecule.get(chembl_id)
        data['max_phase'] = mol.get('max_phase')
        data['drug_type'] = mol.get('molecule_type')

        mech = mechanism.filter(molecule_chembl_id=chembl_id)
        if mech:
            data['mechanism'] = mech[0].get('mechanism_of_action')
            data['target'] = mech[0].get('target_name')
        else:
            data['mechanism'] = data['target'] = None
    except Exception as e:
        print(f"[ChEMBL fail] {name}: {e}")
        data['max_phase'] = data['drug_type'] = data['mechanism'] = data['target'] = None

    time.sleep(0.5)  # Prevent rate limits
    return data

# 7. Loop with caching
for i, name in enumerate(tqdm(drug_names)):
    if name.lower() in done:
        continue
    info = get_drug_info(name)
    results.append(info)

    if len(results) % 100 == 0:
        pd.DataFrame(results).to_csv(CACHE_FILE, index=False)
        print(f"Saved batch at {len(results)} drugs")

# 8. Final save
drug_info_df = pd.DataFrame(results)
drug_info_df.to_csv("final_drug_info.csv", index=False)
print("Enrichment complete. Saved to final_drug_info.csv")

  0%|                                                                               | 1/2176 [00:02<1:36:38,  2.67s/it]

[PubChem fail] sargramostim: list index out of range


  1%|▍                                                                             | 11/2176 [00:27<1:32:23,  2.56s/it]

[PubChem fail] heparin: list index out of range


  1%|▍                                                                             | 12/2176 [00:29<1:23:33,  2.32s/it]

...

[PubChem fail] efpeglenatide: list index out of range


 97%|█████████████████████████████████████████████████████████████████████████▊  | 2113/2176 [1:30:36<02:34,  2.46s/it]

[PubChem fail] actovegin: list index out of range


100%|████████████████████████████████████████████████████████████████████████████| 2176/2176 [1:33:18<00:00,  2.57s/it]

Enrichment complete. Saved to final_drug_info.csv


In [7]:
drug_info_df

,drug_name,smiles,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,target,description
0,succimer,C(C(C(=O)O)S)(C(=O)O)S,"(2S,3R)-2,3-bis(sulfanyl)butanedioic acid",182.2,Succimer; Dim-sa; meso-Dimercaptosuccinic acid...,4.0,Small molecule,Lead chelating agent,None,NaN
1,sargramostim,None,None,None,None,4.0,Protein,Granulocyte-macrophage colony-stimulating fact...,None,NaN
2,caffeine,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,"1,3,7-trimethylpurine-2,6-dione",194.19,"caffeine; 58-08-2; Guaranine; 1,3,7-Trimethylx...",4.0,Small molecule,Adenosine receptor antagonist,None,NaN
3,apixaban,COC1=CC=C(C=C1)N2C3=C(CCN(C3=O)C4=CC=C(C=C4)N5...,1-(4-methoxyphenyl)-7-oxo-6-[4-(2-oxopiperidin...,459.5,Apixaban; 503612-47-3; BMS-562247-01; BMS 5622...,4.0,Small molecule,Coagulation factor X inhibitor,None,NaN
4,ibrexafungerp,CC(C)C(C)C1(CCC2(C3CCC4C5(COCC4(C3=CCC2(C1C(=O...,"(1R,5S,6R,7R,10R,11R,14R,15S,20R,21R)-21-[(2R)...",730.0,Ibrexafungerp; SCY-078; 1207753-03-4; MK-3118;...,4.0,Small molecule,"1,3-beta-glucan synthase inhibitor",None,NaN
...,...,...,...,...,...,...,...,...,...,...
2171,porfiromycin,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2COC(=O)N)OC)N...,"[(4S,6S,7R,8S)-11-amino-7-methoxy-5,12-dimethy...",348.35,Porfiromycin; 801-52-5; N-Methylmitomycin C; M...,3.0,Small molecule,None,None,NaN
2172,penciclovir,C1=NC2=C(N1CCC(CO)CO)N=C(NC2=O)N,2-amino-9-[4-hydroxy-3-(hydroxymethyl)butyl]-1...,253.26,penciclovir; 39809-25-1; Denavir; Pencicloviru...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,None,NaN
2173,cinitapride,CCOC1=CC(=C(C=C1C(=O)NC2CCN(CC2)CC3CCC=CC3)[N+...,4-amino-N-[1-(cyclohex-3-en-1-ylmethyl)piperid...,402.5,Cinitapride; 66564-14-5; Paxapride; Cinitaprid...,3.0,Small molecule,None,None,NaN
2174,melagatran,C1CCC(CC1)C(C(=O)N2CCC2C(=O)NCC3=CC=C(C=C3)C(=...,2-[[(1R)-2-[(2S)-2-[(4-carbamimidoylphenyl)met...,429.5,Melagatran; 159776-70-2; Melagatran [INN]; UNI...,4.0,Small molecule,None,None,NaN


In [8]:
# 9. Merge back to trial data
df_merged = df_expanded.merge(drug_info_df, on='drug_name', how='left').drop_duplicates(subset=['nct_id', 'drug_name'])

def _is_number(val):
    try:
        float(val)
        return True
    except (ValueError, TypeError):
        return False
        
# 10. Collapse per trial
grouped = df_merged.groupby('nct_id').agg({
    'drug_name': lambda x: '; '.join(sorted(set(x.dropna()))),
    'smiles': lambda x: '; '.join(sorted(set(x.dropna()))),
    'description': lambda x: '; '.join(sorted(set(x.dropna()))),
    'iupac': lambda x: '; '.join(sorted(set(x.dropna()))),
    'molecular_weight': lambda x: '; '.join(
        str(round(float(v), 2)) for v in set(x.dropna()) if _is_number(v)
    ),
    'synonyms': lambda x: '; '.join(sorted(set(x.dropna()))),
    'max_phase': lambda x: '; '.join(str(p) for p in set(x.dropna())),
    'drug_type': lambda x: '; '.join(sorted(set(x.dropna()))),
    'mechanism': lambda x: '; '.join(sorted(set(x.dropna()))),
    'target': lambda x: '; '.join(sorted(set(x.dropna()))),
}).reset_index()

In [9]:
grouped

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,target
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,alpha-Tocopherol; alpha Tocopherol; Phytogermi...,4.0,Unknown,,
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Acetamox; Nephramide; ...,4.0,Small molecule,Carbonic anhydrase I inhibitor,
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 5-FU; Fluracil; ...,4.0,Small molecule,Thymidylate synthase inhibitor,
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",255.23; 126.01,foscarnet; Phosphonoformic acid; Carboxyphosph...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",255.23; 126.01,foscarnet; Phosphonoformic acid; Carboxyphosph...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,
...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Selara; SC-66110; CGP-30083; Epler...,4.0; 2.0,Small molecule,Mineralocorticoid receptor antagonist,
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; 4,4'-((1h-1,2,4-triazo...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,


In [11]:
grouped

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,target
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,,
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,
...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",414.5; 380.4,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,


In [16]:
trial_metadata = df.drop_duplicates(subset='nct_id')

# Merge metadata into your grouped dataset (1 row per nct_id)
grouped_enriched = grouped.merge(trial_metadata, on='nct_id', how='left')
grouped_enriched

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,,...,NaN,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,f,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",414.5; 380.4,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,t,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,...,f,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,f,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,...,f,NaN,inclusion criteria:~* participant must be 18 y...,NaN,NaN,t,f,t,single_group treatment none,PHASE1/PHASE2 neoplasms by histologic type; ne...


In [17]:
cols = ['nct_id'] + [col for col in grouped_enriched.columns if col != 'nct_id']
grouped_enriched = grouped_enriched[cols]
grouped_enriched

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,,...,NaN,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,f,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,,"2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",414.5; 380.4,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,t,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,...,f,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,f,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,...,f,NaN,inclusion criteria:~* participant must be 18 y...,NaN,NaN,t,f,t,single_group treatment none,PHASE1/PHASE2 neoplasms by histologic type; ne...


In [18]:
grouped_enriched.to_csv("drugs2smiles.csv", index=False)
print("SAVED")

SAVED


In [35]:
drug_info_df.to_csv("druginfo.csv", index=False)

In [45]:
import pandas as pd

# Load both datasets
druginfo = pd.read_csv("druginfo.csv")
drugbank_info = pd.read_csv("Data/drugbank_drugs_info.csv")

C:\Users\Asus\AppData\Local\Temp\ipykernel_22612\2593809334.py:5: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  drugbank_info = pd.read_csv("Data/drugbank_drugs_info.csv")


In [48]:
druginfo['drug_name'] = druginfo['drug_name'].astype(str).str.lower().str.strip()
drugbank_info['title'] = drugbank_info['title'].astype(str).str.lower().str.strip()

In [49]:
# Merge using 'drug_name' from druginfo and 'title' from drugbank_info
merged = druginfo.merge(
    drugbank_info[['title', 'description']],
    left_on='drug_name',
    right_on='title',
    how='left'
)

merged = merged.drop(columns=['title'])

In [50]:
merged

,drug_name,smiles,iupac,molecular_weight,synonyms,description_x,max_phase,drug_type,mechanism,target,description_y
0,succimer,NaN,NaN,NaN,NaN,NaN,4.0,Small molecule,Lead chelating agent,NaN,"Succimer (2,3-meso-dimercaptosuccinic acid) is..."
1,succimer,NaN,NaN,NaN,NaN,NaN,4.0,Small molecule,Lead chelating agent,NaN,NaN
2,sargramostim,NaN,NaN,NaN,NaN,NaN,4.0,Protein,Granulocyte-macrophage colony-stimulating fact...,NaN,NaN
3,sargramostim,NaN,NaN,NaN,NaN,NaN,4.0,Protein,Granulocyte-macrophage colony-stimulating fact...,NaN,NaN
4,sargramostim,NaN,NaN,NaN,NaN,NaN,4.0,Protein,Granulocyte-macrophage colony-stimulating fact...,NaN,\N
...,...,...,...,...,...,...,...,...,...,...,...
90613,porfiromycin,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2COC(=O)N)OC)N...,"[(4S,6S,7R,8S)-11-amino-7-methoxy-5,12-dimethy...",348.35,Porfiromycin; 801-52-5; N-Methylmitomycin C; M...,NaN,3.0,Small molecule,NaN,NaN,NaN
90614,penciclovir,C1=NC2=C(N1CCC(CO)CO)N=C(NC2=O)N,2-amino-9-[4-hydroxy-3-(hydroxymethyl)butyl]-1...,253.26,penciclovir; 39809-25-1; Denavir; Pencicloviru...,NaN,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,NaN,Penciclovir every 2 hours during waking hours ...
90615,cinitapride,CCOC1=CC(=C(C=C1C(=O)NC2CCN(CC2)CC3CCC=CC3)[N+...,4-amino-N-[1-(cyclohex-3-en-1-ylmethyl)piperid...,402.50,Cinitapride; 66564-14-5; Paxapride; Cinitaprid...,NaN,3.0,Small molecule,NaN,NaN,"cinitapride 1 mg for each dose, 3 mg/daily, fo..."
90616,melagatran,C1CCC(CC1)C(C(=O)N2CCC2C(=O)NCC3=CC=C(C=C3)C(=...,2-[[(1R)-2-[(2S)-2-[(4-carbamimidoylphenyl)met...,429.50,Melagatran; 159776-70-2; Melagatran [INN]; UNI...,NaN,4.0,Small molecule,NaN,NaN,NaN


In [55]:
# 2. Data normalization
druginfo['drug_name'] = druginfo['drug_name'].astype(str).str.lower().str.strip()
drugbank_info['title'] = drugbank_info['title'].astype(str).str.lower().str.strip()

# 3. Remove duplicates 
drugbank_dedup = drugbank_info.drop_duplicates(subset='title')

# 4. Merge
merged = druginfo.merge(
    drugbank_dedup[['title', 'description']],
    left_on='drug_name',
    right_on='title',
    how='left'
)

# 5. Drop column
merged = merged.drop(columns=['title'])
merged

C:\Users\Asus\AppData\Local\Temp\ipykernel_22612\1384930884.py:2: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  drugbank_info = pd.read_csv("Data/drugbank_drugs_info.csv")


,drug_name,smiles,iupac,molecular_weight,synonyms,description_x,max_phase,drug_type,mechanism,target,description_y
0,succimer,NaN,NaN,NaN,NaN,NaN,4.0,Small molecule,Lead chelating agent,NaN,"Succimer (2,3-meso-dimercaptosuccinic acid) is..."
1,sargramostim,NaN,NaN,NaN,NaN,NaN,4.0,Protein,Granulocyte-macrophage colony-stimulating fact...,NaN,NaN
2,caffeine,NaN,NaN,NaN,NaN,NaN,4.0,Small molecule,Adenosine receptor antagonist,NaN,intra-arterial (brachial artery of non dominan...
3,apixaban,NaN,NaN,NaN,NaN,NaN,4.0,Small molecule,Coagulation factor X inhibitor,NaN,"Apixaban: Twice daily, 30 days\nPlacebo: Once ..."
4,ibrexafungerp,NaN,NaN,NaN,NaN,NaN,4.0,Small molecule,"1,3-beta-glucan synthase inhibitor",NaN,Ibrexafungerp 300 mg BID for 1 day
...,...,...,...,...,...,...,...,...,...,...,...
2186,porfiromycin,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2COC(=O)N)OC)N...,"[(4S,6S,7R,8S)-11-amino-7-methoxy-5,12-dimethy...",348.35,Porfiromycin; 801-52-5; N-Methylmitomycin C; M...,NaN,3.0,Small molecule,NaN,NaN,NaN
2187,penciclovir,C1=NC2=C(N1CCC(CO)CO)N=C(NC2=O)N,2-amino-9-[4-hydroxy-3-(hydroxymethyl)butyl]-1...,253.26,penciclovir; 39809-25-1; Denavir; Pencicloviru...,NaN,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,NaN,Penciclovir every 2 hours during waking hours ...
2188,cinitapride,CCOC1=CC(=C(C=C1C(=O)NC2CCN(CC2)CC3CCC=CC3)[N+...,4-amino-N-[1-(cyclohex-3-en-1-ylmethyl)piperid...,402.50,Cinitapride; 66564-14-5; Paxapride; Cinitaprid...,NaN,3.0,Small molecule,NaN,NaN,"cinitapride 1 mg for each dose, 3 mg/daily, fo..."
2189,melagatran,C1CCC(CC1)C(C(=O)N2CCC2C(=O)NCC3=CC=C(C=C3)C(=...,2-[[(1R)-2-[(2S)-2-[(4-carbamimidoylphenyl)met...,429.50,Melagatran; 159776-70-2; Melagatran [INN]; UNI...,NaN,4.0,Small molecule,NaN,NaN,NaN


In [58]:
merged.to_csv("druginfo_with_descriptions.csv", index=False)

In [59]:
merged = merged.drop(columns=['description_x'])

# 2. Rename description
merged = merged.rename(columns={'description_y': 'description'})

In [60]:
merged

,drug_name,smiles,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,target,description
0,succimer,NaN,NaN,NaN,NaN,4.0,Small molecule,Lead chelating agent,NaN,"Succimer (2,3-meso-dimercaptosuccinic acid) is..."
1,sargramostim,NaN,NaN,NaN,NaN,4.0,Protein,Granulocyte-macrophage colony-stimulating fact...,NaN,NaN
2,caffeine,NaN,NaN,NaN,NaN,4.0,Small molecule,Adenosine receptor antagonist,NaN,intra-arterial (brachial artery of non dominan...
3,apixaban,NaN,NaN,NaN,NaN,4.0,Small molecule,Coagulation factor X inhibitor,NaN,"Apixaban: Twice daily, 30 days\nPlacebo: Once ..."
4,ibrexafungerp,NaN,NaN,NaN,NaN,4.0,Small molecule,"1,3-beta-glucan synthase inhibitor",NaN,Ibrexafungerp 300 mg BID for 1 day
...,...,...,...,...,...,...,...,...,...,...
2186,porfiromycin,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2COC(=O)N)OC)N...,"[(4S,6S,7R,8S)-11-amino-7-methoxy-5,12-dimethy...",348.35,Porfiromycin; 801-52-5; N-Methylmitomycin C; M...,3.0,Small molecule,NaN,NaN,NaN
2187,penciclovir,C1=NC2=C(N1CCC(CO)CO)N=C(NC2=O)N,2-amino-9-[4-hydroxy-3-(hydroxymethyl)butyl]-1...,253.26,penciclovir; 39809-25-1; Denavir; Pencicloviru...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,NaN,Penciclovir every 2 hours during waking hours ...
2188,cinitapride,CCOC1=CC(=C(C=C1C(=O)NC2CCN(CC2)CC3CCC=CC3)[N+...,4-amino-N-[1-(cyclohex-3-en-1-ylmethyl)piperid...,402.50,Cinitapride; 66564-14-5; Paxapride; Cinitaprid...,3.0,Small molecule,NaN,NaN,"cinitapride 1 mg for each dose, 3 mg/daily, fo..."
2189,melagatran,C1CCC(CC1)C(C(=O)N2CCC2C(=O)NCC3=CC=C(C=C3)C(=...,2-[[(1R)-2-[(2S)-2-[(4-carbamimidoylphenyl)met...,429.50,Melagatran; 159776-70-2; Melagatran [INN]; UNI...,4.0,Small molecule,NaN,NaN,NaN


In [61]:
# 9. Merge back to trial data
df_merged = df_expanded.merge(merged, on='drug_name', how='left').drop_duplicates(subset=['nct_id', 'drug_name'])

def _is_number(val):
    try:
        float(val)
        return True
    except (ValueError, TypeError):
        return False
        
# 10. Collapse per trial
grouped = df_merged.groupby('nct_id').agg({
    'drug_name': lambda x: '; '.join(sorted(set(x.dropna()))),
    'smiles': lambda x: '; '.join(sorted(set(x.dropna()))),
    'description': lambda x: '; '.join(sorted(set(x.dropna()))),
    'iupac': lambda x: '; '.join(sorted(set(x.dropna()))),
    'molecular_weight': lambda x: '; '.join(
        str(round(float(v), 2)) for v in set(x.dropna()) if _is_number(v)
    ),
    'synonyms': lambda x: '; '.join(sorted(set(x.dropna()))),
    'max_phase': lambda x: '; '.join(str(p) for p in set(x.dropna())),
    'drug_type': lambda x: '; '.join(sorted(set(x.dropna()))),
    'mechanism': lambda x: '; '.join(sorted(set(x.dropna()))),
    'target': lambda x: '; '.join(sorted(set(x.dropna()))),
}).reset_index()

In [62]:
grouped

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,target
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,,
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,Subjects will begin with four 250 mg tablets d...,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,
...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,Fludrocortisone 0.2 mg by mouth daily for 30 d...,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,Letrozole 2.5 mg by mouth daily for 24 weeks,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,


In [66]:
df = pd.read_csv("ctod_data.csv", low_memory=False)
trial_metadata = df.drop_duplicates(subset='nct_id')

# Merge metadata into your grouped dataset (1 row per nct_id)
grouped_enriched = grouped.merge(trial_metadata, on='nct_id', how='left')
grouped_enriched

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,,...,NaN,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,Subjects will begin with four 250 mg tablets d...,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,f,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,Fludrocortisone 0.2 mg by mouth daily for 30 d...,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,t,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,...,f,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,Letrozole 2.5 mg by mouth daily for 24 weeks,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,f,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,...,f,NaN,inclusion criteria:~* participant must be 18 y...,NaN,NaN,t,f,t,single_group treatment none,PHASE1/PHASE2 neoplasms by histologic type; ne...


In [67]:
cols = ['nct_id'] + [col for col in grouped_enriched.columns if col != 'nct_id']
grouped_enriched = grouped_enriched[cols]
grouped_enriched

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,,...,NaN,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,Subjects will begin with four 250 mg tablets d...,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,f,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,Fludrocortisone 0.2 mg by mouth daily for 30 d...,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,t,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,,...,f,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,Letrozole 2.5 mg by mouth daily for 24 weeks,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,f,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...
61522,NCT06655818,dostarlimab,,,,,,4.0,Antibody,Programmed cell death protein 1 antagonist,...,f,NaN,inclusion criteria:~* participant must be 18 y...,NaN,NaN,t,f,t,single_group treatment none,PHASE1/PHASE2 neoplasms by histologic type; ne...


In [68]:
grouped_enriched.to_csv("drugbank_info_smiles.csv", index=False)
print("SAVED")

SAVED
